# Step 2: Fix the binary header and convert g1/g2/g3 to x/y/z

This notebook belongs to the Geo-OBS manual, Session 6 ("Binary Header Correction" and "X, Y, Z Conversion").

**Input:** the SEG-Y files exported by EDR, one set per chunk folder:

```
<DATA_FOLDER>/<station>/1stChunk/<station>_1stChunk_seq1.g1_segy   (geophone 1)
                                  ..._seq1.g2_segy                  (geophone 2)
                                  ..._seq1.g3_segy                  (geophone 3)
                                  ..._seq1.h1_segy                  (hydrophone)
<DATA_FOLDER>/<station>/2ndChunk/...
```

**Output**, written into the same chunk folder:

```
<station>_1stChunk_x.segy   <station>_1stChunk_y.segy   <station>_1stChunk_z.segy   <station>_1stChunk_h.segy
```

**What it does**

1. EDR writes a few wrong numbers into the binary header (for example `-2768` samples per trace instead of `30000`). GeoGiga and other programs cannot open the files. The notebook writes the correct values.
2. The three geophones of the Trilobit are tilted (Galperin configuration). The notebook converts them to x, y and z (z positive down).
3. The hydrophone only gets the header fix.
4. It checks the new files. Only if the check passes **and** you set `DELETE_RAW_FILES = True`, the EDR files are deleted.

Run `1_check_edr_export.ipynb` first. Run the cells from top to bottom (menu *Run > Run All Cells*). You only have to change the cell **Settings**.

**Time and memory:** about 10 to 20 minutes per station on an external hard drive (the drive speed is the limit). The files are processed in small blocks, so a normal laptop is enough.

## Settings

In [ ]:
from pathlib import Path

DATA_FOLDER = Path("/Volumes/NilRodes/zJoshi")   # folder that contains one folder per station
STATIONS = ["0989"]                               # e.g. ["0498", "0578"] for several stations

# Delete the EDR files (g1/g2/g3/h1) after a successful check?
# Leave False for the first run and look at the result in step 4 (QC) before you delete anything.
DELETE_RAW_FILES = False

## Recording parameters

These are fixed for the Trilobit recordings of this course (2 ms sampling, 60 s per trace). Do not change them unless your EDR export settings were different.

In [ ]:
import re
import struct
import numpy as np
from datetime import datetime, timedelta

SAMPLE_INTERVAL_US = 2000     # 2 ms = 500 Hz
SAMPLES_PER_TRACE = 30000     # 60 s per trace

HEADER_BYTES = 3600           # 3200 byte text header + 400 byte binary header
TRACE = np.dtype([("header", "V240"), ("data", ">f4", SAMPLES_PER_TRACE)])   # one trace: 240 byte header + samples
TRACES_PER_BLOCK = 200        # traces read at once (about 25 MB per file)

RAW_NAMES = {"g1": "g1_segy", "g2": "g2_segy", "g3": "g3_segy", "h": "h1_segy"}

## 1. The binary header fix

The binary header is the 400 bytes after the text header (bytes 3201 to 3600). The table shows the fields that the notebook writes. Byte positions follow the SEG-Y standard (rev 1).

| Bytes | Field | Value written |
|---|---|---|
| 3217-3218 | sample interval [µs] | 2000 |
| 3219-3220 | sample interval of the original recording [µs] | 2000 |
| 3221-3222 | samples per trace | 30000 |
| 3223-3224 | samples per trace of the original recording | 30000 |
| 3227-3228 | ensemble fold | 1 |
| 3229-3230 | trace sorting code | 1 (as recorded) |
| 3231-3232 | vertical sum code | 1 (no sum) |
| 3235-3236 | sweep frequency at end [Hz] | 0 (no sweep) |
| 3255-3256 | measurement system | 1 (meters) |

The data format code (bytes 3225-3226) is already correct (5 = 4-byte IEEE float) and is not touched. Text header, trace headers and samples are copied unchanged.

In [ ]:
def fixed_header(raw_path):
    """Read the 3600 byte file header of an EDR file and return it with the corrected binary header."""
    with open(raw_path, "rb") as f:
        header = bytearray(f.read(HEADER_BYTES))
    corrections = {
        3217: SAMPLE_INTERVAL_US,
        3219: SAMPLE_INTERVAL_US,
        3221: SAMPLES_PER_TRACE,
        3223: SAMPLES_PER_TRACE,
        3227: 1,
        3229: 1,
        3231: 1,
        3235: 0,
        3255: 1,
    }
    for byte, value in corrections.items():
        position = byte - 1                      # bytes are counted from 1 in the SEG-Y standard
        header[position:position + 2] = struct.pack(">h", value)
    return header

## 2. Galperin to x, y, z

Each geophone is tilted by 54.7° from the vertical, and the three are 120° apart. The conversion is the matrix from the manufacturer's note (*Trilobit geophone configurations*, Carrack 2010):

$$
\begin{pmatrix} x \\ y \\ z \end{pmatrix} =
\begin{pmatrix}
-\sin 54.7^\circ & \sin 54.7^\circ \cos 60^\circ & \sin 54.7^\circ \cos 60^\circ \\
0 & -\sin 54.7^\circ \cos 30^\circ & \sin 54.7^\circ \cos 30^\circ \\
\cos 54.7^\circ & \cos 54.7^\circ & \cos 54.7^\circ
\end{pmatrix}
\begin{pmatrix} g_1 \\ g_2 \\ g_3 \end{pmatrix}
$$

x points along geophone 1, z is positive **down** (seismic convention), and x, y, z form a right-handed system. x and y are still in the frame of the instrument, not north and east.

In [ ]:
def galperin_to_xyz(g1, g2, g3):
    x = -0.81650 * g1 + 0.40825 * (g2 + g3)
    y = 0.70711 * (g3 - g2)
    z = 0.57735 * (g1 + g2 + g3)
    return x, y, z

## 3. Helper functions

`find_chunk_folders` finds `1stChunk`, `2ndChunk`, ... (and ignores folders like `DailyChunk`). `trace_times` reads the start time of the first and the last trace, which is used for the check.

In [ ]:
def find_chunk_folders(station_folder):
    return sorted(p for p in station_folder.iterdir()
                  if p.is_dir() and re.match(r"^\d+(st|nd|rd|th)Chunk$", p.name))


def number_of_traces(path):
    return (path.stat().st_size - HEADER_BYTES) // TRACE.itemsize


def trace_start_time(trace_header):
    year, day, hour, minute, second = struct.unpack(">5H", trace_header[156:166])
    millisecond, microsecond = struct.unpack(">2H", trace_header[168:172])
    return datetime(year, 1, 1) + timedelta(days=day - 1, hours=hour, minutes=minute, seconds=second,
                                            milliseconds=millisecond, microseconds=microsecond)


def trace_times(path):
    """Start time of the first and of the last trace."""
    n = number_of_traces(path)
    with open(path, "rb") as f:
        f.seek(HEADER_BYTES)
        first = trace_start_time(f.read(240))
        f.seek(HEADER_BYTES + (n - 1) * TRACE.itemsize)
        last = trace_start_time(f.read(240))
    return first, last

## 4. Convert one chunk folder

The four EDR files are read block by block (`TRACES_PER_BLOCK` traces at a time), converted and written. The new files first get the ending `.part` and are only renamed when they are complete, so an interrupted run never leaves a half file with a correct name. After writing, `check_outputs` compares every output with its input: same size, same number of traces, same first and last trace time.

In [ ]:
def chunk_files(chunk_folder):
    stem = f"{chunk_folder.parent.name}_{chunk_folder.name}"
    raw = {c: chunk_folder / f"{stem}_seq1.{ext}" for c, ext in RAW_NAMES.items()}
    out = {c: chunk_folder / f"{stem}_{c}.segy" for c in "xyzh"}
    return raw, out


def convert_chunk(raw, out):
    n = number_of_traces(raw["g1"])
    for p in raw.values():
        if (p.stat().st_size - HEADER_BYTES) % TRACE.itemsize != 0:
            print(f"  WARNING {p.name}: size is not a whole number of traces - check SAMPLES_PER_TRACE")
        if number_of_traces(p) != n:
            raise ValueError(f"{p.name} has {number_of_traces(p)} traces, {raw['g1'].name} has {n}")

    part = {c: p.with_name(p.name + ".part") for c, p in out.items()}
    inputs = {c: open(p, "rb") for c, p in raw.items()}
    outputs = {c: open(p, "wb") for c, p in part.items()}
    try:
        for c in "xyz":
            outputs[c].write(fixed_header(raw["g1"]))     # x, y and z use the file header of g1
        outputs["h"].write(fixed_header(raw["h"]))
        for f in inputs.values():
            f.seek(HEADER_BYTES)

        done = 0
        while done < n:
            count = min(TRACES_PER_BLOCK, n - done)
            g1, g2, g3, h = (np.fromfile(inputs[c], dtype=TRACE, count=count) for c in ("g1", "g2", "g3", "h"))
            x, y, z = galperin_to_xyz(g1["data"].astype(np.float64), g2["data"], g3["data"])
            for c, samples in zip("xyz", (x, y, z)):
                block = g1.copy()                         # keep the trace headers of g1
                block["data"] = samples
                block.tofile(outputs[c])
            h.tofile(outputs["h"])                        # hydrophone: traces unchanged
            done += count
            if done % 2000 < TRACES_PER_BLOCK or done == n:
                print(f"  {done} / {n} traces")
    finally:
        for f in list(inputs.values()) + list(outputs.values()):
            f.close()

    for c in "xyzh":
        part[c].replace(out[c])


def check_outputs(raw, out):
    """Every output must have the same size and the same first/last trace time as its EDR file."""
    source = {"x": "g1", "y": "g1", "z": "g1", "h": "h"}
    ok = True
    for c, r in source.items():
        same_size = out[c].stat().st_size == raw[r].stat().st_size
        same_times = trace_times(out[c]) == trace_times(raw[r])
        if not (same_size and same_times):
            print(f"  CHECK FAILED {out[c].name}: same size {same_size}, same trace times {same_times}")
            ok = False
    return ok

## 5. Run

For every chunk folder:

- EDR files present, no x/y/z/h yet: convert, check, delete the EDR files if `DELETE_RAW_FILES = True`.
- EDR files and x/y/z/h present (second run): only check, then delete the EDR files if `DELETE_RAW_FILES = True`.
- Only x/y/z/h present: nothing to do.

In [ ]:
for station in STATIONS:
    chunk_folders = find_chunk_folders(DATA_FOLDER / station)
    if not chunk_folders:
        print(f"{station}: no chunk folders (1stChunk, 2ndChunk, ...) found in {DATA_FOLDER / station}")

    for chunk_folder in chunk_folders:
        name = f"{station} {chunk_folder.name}"
        raw, out = chunk_files(chunk_folder)
        raw_exists = all(p.exists() for p in raw.values())
        out_exists = all(p.exists() for p in out.values())

        if not raw_exists:
            if out_exists:
                print(f"{name}: already converted, EDR files already deleted")
            else:
                print(f"{name}: EDR files missing: {[p.name for p in raw.values() if not p.exists()]}")
            continue

        if out_exists and check_outputs(raw, out):
            print(f"{name}: already converted")
        else:
            print(f"{name}: converting ...")
            convert_chunk(raw, out)

        first, last = trace_times(out["x"])
        print(f"  {number_of_traces(out['x'])} traces, {first} to {last + timedelta(seconds=60)} UTC")
        if check_outputs(raw, out):
            print("  check OK")
            if DELETE_RAW_FILES:
                for p in raw.values():
                    p.unlink()
                print("  EDR files deleted")
        else:
            print("  EDR files kept because the check failed")

## Next step

Cut the x/y/z/h files into daily files (`3_cut_daily_files.ipynb`) and look at them with `4_qc_daily_files.ipynb`, or open one `_z.segy` file in GeoGiga. If everything looks fine and you need the disk space, set `DELETE_RAW_FILES = True` and run this notebook again: converted chunks are only checked, and then the EDR files are deleted.